# Spotting a Leaky Predictor

### A subscription model that looks too good to be true

A bank marketing team phones customers to sell a term deposit. A data analyst has built a model that predicts who will subscribe, and it scores brilliantly. Before the team plans a campaign around it, you are asked one question: **could the bank actually use this model to decide whom to ring?**

**How this notebook works**

- Five steps: load and look, split and prepare, fit with everything, fit without the suspect column, decide and write the note.
- Python leads and JMP is the check. If the two disagree, question the Python first.
- GitHub Copilot is on. Use it, and check what it writes against your JMP result before you believe it.
- Run cells in order with **Shift+Enter**. The data files live in `../data/`.
- The file holds 4,521 contacts from a bank's phone campaign. `subscribed` is `yes` for the 11.5% who took the deposit.

---
## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix

GREEN, AMBER = "#1A4C3B", "#E07B39"

print(sorted(p.name for p in Path("../data").iterdir()))


def odds_ratio_table(model):
    """Odds ratios with 95% intervals and p-values for a fitted statsmodels Logit."""
    ci = np.exp(model.conf_int())
    table = pd.DataFrame({"odds ratio": np.exp(model.params), "95% low": ci[0], "95% high": ci[1], "p-value": model.pvalues})
    table["p-value"] = ["<0.0001" if p < 0.0001 else f"{p:.4f}" for p in table["p-value"]]
    return table.round({"odds ratio": 3, "95% low": 3, "95% high": 3})


def what_if(model, base, changes, base_label, new_label, outcome, digits=1):
    """Change something about a base case and show the chance of the outcome before and after."""
    def chance(row):
        x = pd.DataFrame([row])[list(model.params.index[1:])]
        return float(np.asarray(model.predict(sm.add_constant(x, has_constant="add")))[0])

    new = base.copy()
    for column, value in changes.items():
        new[column] = value
    p_base, p_new = chance(base), chance(new)
    high, low = max(p_base, p_new), min(p_base, p_new)
    high_label, low_label = (new_label, base_label) if p_new >= p_base else (base_label, new_label)
    odds_ratio = (high / (1 - high)) / (low / (1 - low))
    chance_ratio = high / low
    print(f"{base_label}: {p_base:.1%} likely to {outcome}")
    print(f"{new_label}: {p_new:.1%}")
    print(f"Odds ratio ({high_label} against {low_label}): {odds_ratio:.{digits}f}")
    print(f"Ratio of the probabilities ({high_label} against {low_label}): {chance_ratio:.{digits}f}")
    print(f"For a manager: {high_label} has {odds_ratio:.{digits}f} times the odds, but only "
          f"{chance_ratio:.{digits}f} times the chance, of the outcome compared with {low_label}.")
    return p_base, p_new

---
## Step 1: load and look

**The business question:** the team wants to ring the customers most likely to subscribe. Which customers should it ring, and could it have known that before dialling?

In [ ]:
bank = pd.read_excel("../data/BankSample.xlsx", sheet_name="BankSample")
print(bank.shape)
print(f"Share who subscribed: {(bank['subscribed'] == 'yes').mean():.1%}")
bank.head()

In [ ]:
# How long did the calls last for those who subscribed and those who did not? (seconds)
print(bank.groupby("subscribed")["duration"].mean().round(0))

Customers who subscribe talk to the bank for much longer than those who do not. That sounds useful. Hold that thought: when is the length of a call known?

## Step 2: split first and prepare

The same split call as every other model on this module. This file has no blanks to fill, so there are no medians to learn, but the split still comes first, as a habit that stays safe when there are. Two columns are rescaled so that an odds ratio is for a sensible change: `duration` in minutes and `balance` in thousands.

In [ ]:
# Rescale two columns, then split with the same call as always
bank_model = bank.drop(columns="subscribed").copy()
bank_model["duration"] = bank_model["duration"] / 60          # seconds to minutes
bank_model["balance"] = bank_model["balance"] / 1000          # to thousands
y_bank = (bank["subscribed"] == "yes").astype(int)

X_train, X_test, y_train, y_test = train_test_split(bank_model, y_bank, test_size=0.3, random_state=1, stratify=y_bank)
print(f"Train: {len(X_train):,} contacts, {y_train.sum()} subscribed ({y_train.mean():.2%})")
print(f"Test:  {len(X_test):,} contacts, {y_test.sum()} subscribed ({y_test.mean():.2%})")

In [ ]:
# Text columns become 0/1 indicators. The reference level for each is the most common one in the TRAINING rows.
categories = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]
reference = {c: X_train[c].value_counts().index[0] for c in categories}
print(reference)

def indicators(frame):
    out = pd.get_dummies(frame, columns=categories, dtype=int)
    return out.drop(columns=[f"{c}_{reference[c]}" for c in categories])

train_readable, test_readable = X_train.copy(), X_test.copy()      # kept for the JMP export
X_train = indicators(X_train)
X_test = indicators(X_test).reindex(columns=X_train.columns, fill_value=0)
print(X_train.shape)

## Step 3: fit with everything

In [ ]:
# Fit on the training contacts with every column, then score the test contacts
all_cols = list(X_train.columns)
model_all = sm.Logit(y_train, sm.add_constant(X_train[all_cols])).fit(disp=False, maxiter=200)
p_all = model_all.predict(sm.add_constant(X_test[all_cols], has_constant="add"))

auc_all = roc_auc_score(y_test, p_all)
print(f"Converged: {model_all.mle_retvals['converged']}")
print(f"Test AUC with every column: {auc_all:.3f}")

**Pause and answer before you read on.** An AUC of 0.883 is high for a model that predicts who says yes to a phone call. Is it believable? Write down what you would want to check about the columns the model used.

In [ ]:
# The confusion matrix at a cut-off of 0.5. Positive means "subscribes".
def confusion(y_true, p, cutoff=0.5):
    tn, fp, fn, tp = confusion_matrix(y_true, (np.asarray(p) >= cutoff).astype(int)).ravel()
    return pd.DataFrame([[tn, fp], [fn, tp]], index=["Did not subscribe", "Subscribed"],
                        columns=["Model says no", "Model says yes"])

print(confusion(y_test, p_all))
accuracy_all = ((p_all >= 0.5).astype(int) == y_test).mean()
print(f"\nAccuracy: {accuracy_all:.1%}   Accuracy of predicting that nobody subscribes: {1 - y_test.mean():.1%}")

In [ ]:
# The five columns with the smallest p-values. The call length leads the table.
table_all = odds_ratio_table(model_all).drop(index="const")
table_all.assign(p_numeric=model_all.pvalues.drop("const")).sort_values("p_numeric").drop(columns="p_numeric").head(5)

Look at the first row. Each extra minute on the phone multiplies the odds of subscribing by about 1.3. The model is learning that long calls end in a sale. A what-if shows how much that is worth:

In [ ]:
# A typical contact: number columns at the training median, every indicator off (the most common level of each)
typical = pd.Series(0.0, index=X_train.columns)
for column in ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]:
    typical[column] = X_train[column].median()

_ = what_if(model_all, typical, {"duration": typical["duration"] + 5}, "Typical contact",
        "Same contact with a call five minutes longer", "subscribe", digits=1)

## Step 4: fit without the suspect column

Now refit without `duration` and compare. Everything else stays the same: the same split, the same other columns.

In [ ]:
no_duration = [c for c in all_cols if c != "duration"]
model_nd = sm.Logit(y_train, sm.add_constant(X_train[no_duration])).fit(disp=False, maxiter=200)
p_nd = model_nd.predict(sm.add_constant(X_test[no_duration], has_constant="add"))

auc_nd = roc_auc_score(y_test, p_nd)
print(f"Test AUC without duration: {auc_nd:.3f}   (with it: {auc_all:.3f})")
print(confusion(y_test, p_nd))
accuracy_nd = ((p_nd >= 0.5).astype(int) == y_test).mean()
print(f"\nAccuracy: {accuracy_nd:.1%}   Accuracy of predicting that nobody subscribes: {1 - y_test.mean():.1%}")

In [ ]:
# Side by side
pd.DataFrame({
    "test AUC": [auc_all, auc_nd],
    "accuracy at 0.5": [accuracy_all, accuracy_nd],
    "subscribers caught": [int(((p_all >= 0.5) & (y_test == 1)).sum()), int(((p_nd >= 0.5) & (y_test == 1)).sum())],
    "out of": [int(y_test.sum())] * 2,
}, index=["with duration", "without duration"]).round(3)

### Commentary

The model with `duration` scores an AUC of 0.883. Why is that too good to be true, and what does the lower AUC of 0.741 tell the bank?

*Your commentary here*

## Step 5: decide what a campaign could use, then write the note

A predictor can only be used if its value is known **at the moment of the decision**, here before anyone is rung. For each column below, mark whether it is known before the call: `"yes"`, `"no"`, or `"unsure"` if it depends on how the bank recorded it. The columns that describe the contact itself (how many times the customer was contacted in this campaign, and the day and month of the contact) are worth a second look.

In [ ]:
# Fill in "yes", "no" or "unsure" for each column
known_before_call = {
    "age": None, "job": None, "marital": None, "education": None, "default": None,
    "balance": None, "housing": None, "loan": None, "contact": None,
    "day": None, "month": None, "duration": None, "campaign": None,
    "pdays": None, "previous": None, "poutcome": None,
}

In [ ]:
# Which columns did you flag, and what happens to the test AUC if they all go?
flagged = [c for c, v in known_before_call.items() if v in ("no", "unsure")]
print("Columns you would not rely on before a call:", flagged)

# Drop the flagged columns, including every indicator built from them
keep = [c for c in all_cols if not any(c == f or c.startswith(f + "_") for f in flagged)]
if flagged:
    model_safe = sm.Logit(y_train, sm.add_constant(X_train[keep])).fit(disp=False, maxiter=200)
    auc_safe = roc_auc_score(y_test, model_safe.predict(sm.add_constant(X_test[keep], has_constant="add")))
    print(f"Test AUC using only the columns you trust: {auc_safe:.3f}")

### Check against JMP

Export the prepared table with the `Split` column and fit the same two models in JMP, with and without `duration`, following the brief. Set the Target Level to `yes`.

In [ ]:
bank_export = pd.concat([
    train_readable.assign(subscribed=np.where(y_train == 1, "yes", "no"), Split="Train", Validation=0),
    test_readable.assign(subscribed=np.where(y_test == 1, "yes", "no"), Split="Test", Validation=1),
]).sort_index()
front = ["subscribed", "Split", "Validation"]
bank_export = bank_export[front + [c for c in bank_export.columns if c not in front]]
bank_export.to_csv("../data/bank_prepared_with_split.csv", index=False)
print(bank_export["Split"].value_counts())

# Your JMP numbers go here
jmp = {"Test AUC with duration": None, "Test AUC without duration": None}
python = {"Test AUC with duration": auc_all, "Test AUC without duration": auc_nd}
for name, value in python.items():
    other = jmp[name]
    verdict = "not recorded" if other is None else f"match: {np.isclose(other, value, atol=0.01)}"
    print(f"{name:28} Python {value:.3f}   JMP {other}   {verdict}")

### Say it to a manager

A note of **150 words or fewer** to the head of marketing, saved in your repository. It must say, in plain English: whether the model with `duration` can be used and why not; what a campaign could expect from the honest model; and **which predictors would be known before the call**. Use a what-if, not a coefficient: for example, what a previous successful campaign does to the chance of subscribing for a typical customer.

The translator gives you raw material:

In [ ]:
# A what-if that only uses information known before the call: a previous campaign that succeeded
_ = what_if(model_nd, typical.drop("duration"), {"poutcome_success": 1}, "Typical contact", "Same contact whose previous campaign succeeded", "subscribe", digits=1)

*Your note here (150 words or fewer)*

---
## Stretch

A blank space for quick work. Use Copilot to take the analysis further. Some ideas: choose a cut-off from invented costs (a call costs 5, a subscription is worth 100) and find the profit on the test contacts; compare the AUC when you also drop `campaign`, `day` and `month`; or find which month has the highest subscription rate and ask whether that is a reason to ring in that month.